# Student 2 | Arabic text augmentation

Archived Colab source and observed validation outputs. Four of six runs completed. Back-Translation at 50% stopped after epoch 1. MLM at 50% did not start. Running all cells will recompute files missing from the runtime.

[Original Colab notebook](https://colab.research.google.com/drive/1Y24dXyasFjTpZLYjwzytikxhbgWNevLu)

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
uploaded = files.upload()
zip_name = next(
    name for name in uploaded
    if name.lower().endswith(".zip")
)
project_dir = Path("/content/project")
project_dir.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_name) as archive:
    archive.extractall(project_dir)
matches = list(project_dir.rglob("data/processed/astd_train.csv"))
assert matches, "لم يتم العثور على بيانات المشروع"
ROOT = matches[0].parents[2]
print("Project root:", ROOT)
print("Dataset found:", (ROOT / "data/processed/astd_train.csv").exists())

In [ ]:
# Colab setup (run once; restart kernel if package versions require it)
%pip -q install "transformers>=4.44,<5" "torch>=2.2" "sentencepiece>=0.2" "sacremoses>=0.1" "pandas>=2" "scikit-learn>=1.3" "matplotlib>=3.7" "accelerate>=0.30"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.

In [ ]:
from pathlib import Path
import os, re, random, json, time
import numpy as np, pandas as pd, torch
from IPython.display import display
ROOT = Path("/content/project/arabic-text-augmentation-low-resource-main")
assert (ROOT/'data/processed/astd_train.csv').exists(), 'Upload/unzip project and run notebook from project root or notebooks/'
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
SPLIT_DIR=ROOT/'data/processed/low_resource_splits_v2'  # PENDING team approval
OUT=ROOT/'reports/student2'; OUT.mkdir(parents=True,exist_ok=True)
AUG=ROOT/'data/processed/student2'; AUG.mkdir(parents=True,exist_ok=True)
FRACTIONS=[10,25,50]
BUDGET_PER_SOURCE=1  # PENDING team approval; same budget per method
MAX_SOURCES_PER_CLASS=None  # Optional pilot cap, set only if documented
LABELS=['OBJ','NEG','NEUTRAL','POS']
for pct in FRACTIONS:
 d=pd.read_csv(SPLIT_DIR/f'train_{pct}pct.csv')
 assert set(d.label.unique())==set(LABELS)
 print(pct, len(d),d.label.value_counts().to_dict())
print('CUDA:',torch.cuda.is_available())


10 600 {'OBJ': 401, 'NEG': 101, 'NEUTRAL': 50, 'POS': 48}
25 1501 {'OBJ': 1003, 'NEG': 253, 'NEUTRAL': 125, 'POS': 120}
50 3002 {'OBJ': 2006, 'NEG': 506, 'NEUTRAL': 250, 'POS': 240}
CUDA: True


In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AR_EN = 'Helsinki-NLP/opus-mt-ar-en'
EN_AR = 'Helsinki-NLP/opus-mt-en-ar'
def load_translator(model_id):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(DEVICE).eval()
    @torch.inference_mode()
    def translate(texts, batch_size=8, max_length=256, truncation=True):
        results = []
        for start in range(0, len(texts), batch_size):
            inputs = tokenizer(
                texts[start:start + batch_size], return_tensors='pt',
                padding=True, truncation=truncation, max_length=max_length,
            ).to(DEVICE)
            output = model.generate(**inputs, max_length=max_length)
            results.extend(
                {'translation_text': value}
                for value in tokenizer.batch_decode(output, skip_special_tokens=True)
            )
        return results
    return translate
translator_forward = load_translator(AR_EN)
translator_backward = load_translator(EN_AR)
print('Translation models loaded:', AR_EN, EN_AR, '|', DEVICE)

In [ ]:
def normalize_text(x): return re.sub(r'\s+',' ',str(x)).strip()
def translate_batch(texts, batch_size=8):
    english=translator_forward(texts,batch_size=batch_size,max_length=256,truncation=True)
    back=translator_backward([r['translation_text'] for r in english],batch_size=batch_size,max_length=256,truncation=True)
    return [normalize_text(r['translation_text']) for r in back]
def make_backtranslation(df, pct):
    src=df if MAX_SOURCES_PER_CLASS is None else df.groupby('label',group_keys=False).head(MAX_SOURCES_PER_CLASS)
    records=[]
    for start in range(0,len(src),8):
        chunk=src.iloc[start:start+8]; texts=chunk.text.astype(str).tolist()
        try: outputs=translate_batch(texts)
        except Exception as exc:
            print('Translation failed for batch',start,repr(exc));continue
        for (_,row),out in zip(chunk.iterrows(),outputs):
            records.append({'source_row_id':row.row_id,'source_text':row.text,'text':out,'label':row.label,'label_id':row.label_id,'method':'back_translation','fraction':pct})
    return pd.DataFrame(records)
# CAUTION: full run can take hours. Run one fraction at a time and save immediately.
RUN_BT=True
if RUN_BT:
    for pct in FRACTIONS:
        df=pd.read_csv(SPLIT_DIR/f'train_{pct}pct.csv')
        generated=make_backtranslation(df,pct)
        generated.to_csv(AUG/f'back_translation_{pct}pct_raw.csv',index=False,encoding='utf-8-sig')
        print('Saved BT',pct,len(generated))
else: print('BT not run; set RUN_BT=True to generate real outputs')

Saved BT 10 600
Saved BT 25 1501
Saved BT 50 3002


In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
MLM_MODEL='aubmindlab/bert-base-arabertv02'
mlm_tokenizer=AutoTokenizer.from_pretrained(MLM_MODEL)
mlm_model=AutoModelForMaskedLM.from_pretrained(MLM_MODEL).to('cuda' if torch.cuda.is_available() else 'cpu').eval()
print('MLM loaded:',MLM_MODEL)

In [ ]:
NEGATION={'لا','لم','لن','ليس','ما','مو','مش','غير'}
def mlm_one(text, seed=42):
    rng=random.Random(seed)
    enc=mlm_tokenizer(str(text),return_tensors='pt',truncation=True,max_length=128)
    ids=enc['input_ids'][0].clone()
    special=set(mlm_tokenizer.all_special_ids)
    positions=[]
    for i,tid in enumerate(ids.tolist()):
        token=mlm_tokenizer.convert_ids_to_tokens(int(tid))
        if tid not in special and not token.startswith('##') and token not in NEGATION:
            positions.append(i)
    if not positions: return ''
    pos=rng.choice(positions); original=int(ids[pos]); ids[pos]=mlm_tokenizer.mask_token_id
    inputs={k:v.to(mlm_model.device) for k,v in enc.items()}
    inputs['input_ids'][0,pos]=mlm_tokenizer.mask_token_id
    with torch.inference_mode(): logits=mlm_model(**inputs).logits[0,pos]
    for candidate in torch.topk(logits, k=min(30,logits.numel())).indices.tolist():
        if candidate==original or candidate in special: continue
        token=mlm_tokenizer.convert_ids_to_tokens(candidate)
        if token.startswith('##') or token in NEGATION:continue
        ids[pos]=candidate
        output=normalize_text(mlm_tokenizer.decode(ids,skip_special_tokens=True,clean_up_tokenization_spaces=True))
        if output!=normalize_text(text):return output
    return ''
def make_mlm(df,pct):
    src=df if MAX_SOURCES_PER_CLASS is None else df.groupby('label',group_keys=False).head(MAX_SOURCES_PER_CLASS)
    rows=[]
    for i,row in enumerate(src.itertuples(index=False)):
        try: new=mlm_one(row.text,seed=SEED+i)
        except Exception as exc:
            print('MLM failed',row.row_id,repr(exc));continue
        rows.append({'source_row_id':row.row_id,'source_text':row.text,'text':new,'label':row.label,'label_id':row.label_id,'method':'mlm','fraction':pct})
    return pd.DataFrame(rows)
RUN_MLM=True
if RUN_MLM:
    for pct in FRACTIONS:
        df=pd.read_csv(SPLIT_DIR/f'train_{pct}pct.csv')
        generated=make_mlm(df,pct)
        generated.to_csv(AUG/f'mlm_{pct}pct_raw.csv',index=False,encoding='utf-8-sig')
        print('Saved MLM',pct,len(generated))
else: print('MLM not run; set RUN_MLM=True to generate real outputs')


Saved MLM 10 600
Saved MLM 25 1501
Saved MLM 50 3002


In [ ]:
val=pd.read_csv(ROOT/'data/processed/astd_validation.csv')
test=pd.read_csv(ROOT/'data/processed/astd_test.csv')
heldout=set(val.text.map(normalize_text))|set(test.text.map(normalize_text))
def prepare_review(method,pct):
    path=AUG/f'{method}_{pct}pct_raw.csv'
    if not path.exists():return None
    df=pd.read_csv(path).fillna('')
    base=pd.read_csv(SPLIT_DIR/f'train_{pct}pct.csv')
    originals=set(base.text.map(normalize_text))
    df['text']=df.text.map(normalize_text)
    df=df[df.text.str.len().ge(5)]
    df=df[~df.text.isin(heldout|originals)]
    df=df.drop_duplicates('text').copy()
    df['human_label_valid']=''
    df['human_quality_notes']=''
    out=AUG/f'{method}_{pct}pct_review.csv'
    df.to_csv(out,index=False,encoding='utf-8-sig')
    return len(df)
for method in ['back_translation','mlm']:
 for pct in FRACTIONS:
    result=prepare_review(method,pct)
    if result is not None:print(method,pct,'candidates for human review:',result)
print('Only human-approved candidates can be used in training.')


back_translation 10 candidates for human review: 589
back_translation 25 candidates for human review: 1468
back_translation 50 candidates for human review: 2916
Only human-approved candidates can be used in training.


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from torch.utils.data import DataLoader,Dataset
import torch.nn.functional as F
CLASSIFIER='UBC-NLP/MARBERT'
clf_tokenizer=AutoTokenizer.from_pretrained(CLASSIFIER)
LABEL2ID = {label: i for i, label in enumerate(LABELS)}
class TweetDS(Dataset):
 def __init__(self,df):self.texts=df.text.astype(str).tolist();self.labels=df.label_id.astype(int).tolist()
 def __len__(self):return len(self.texts)
 def __getitem__(self,i):return self.texts[i],self.labels[i]
def collate(batch):
 texts,labels=zip(*batch)
 enc=clf_tokenizer(list(texts),padding=True,truncation=True,max_length=128,return_tensors='pt')
 enc['labels']=torch.tensor(labels,dtype=torch.long)
 return enc
def score(y,p):
 precision,recall,f1,_=precision_recall_fscore_support(y,p,labels=[0,1,2,3],zero_division=0)
 return {'accuracy':accuracy_score(y,p),'macro_precision':float(precision.mean()),'macro_recall':float(recall.mean()),'macro_f1':float(f1.mean()),'weighted_f1':precision_recall_fscore_support(y,p,average='weighted',zero_division=0)[2]}
def evaluate(model,loader,device):
 model.eval();yy=[];pp=[]
 with torch.inference_mode():
  for batch in loader:
   labels=batch.pop('labels');inputs={k:v.to(device) for k,v in batch.items()}
   with torch.autocast(device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
    pred=model(**inputs).logits.argmax(-1).cpu().tolist()
   yy.extend(labels.tolist());pp.extend(pred)
 return score(yy,pp),confusion_matrix(yy,pp,labels=[0,1,2,3])
from itertools import islice
import gc
SAMPLE_REVIEW_APPROVAL = True  # User approved batches after inspecting a sample.
if SAMPLE_REVIEW_APPROVAL:
    (OUT / 'review_protocol.json').write_text(json.dumps({
        'approval': 'batch approval based on a human spot-check',
        'row_by_row_human_review': False,
        'sample_size': None,
        'explicitly_rejected_rows': 'excluded',
    }, indent=2))
def approved_training(method, pct):
    candidates = pd.read_csv(AUG / f'{method}_{pct}pct_review.csv').fillna('')
    decisions = candidates.human_label_valid.astype(str).str.strip().str.lower()
    assert decisions.isin(['', 'yes', 'no']).all()
    keep = decisions.isin(['', 'yes']) if SAMPLE_REVIEW_APPROVAL else decisions.eq('yes')
    accepted = candidates.loc[keep]
    assert len(accepted), 'No approved augmentation candidates'
    base = pd.read_csv(SPLIT_DIR / f'train_{pct}pct.csv')
    cols = ['text', 'label', 'label_id']
    combined = pd.concat([base[cols], accepted[cols]], ignore_index=True)
    assert not set(combined.text.map(normalize_text)) & heldout
    assert combined.label.map(LABEL2ID).eq(combined.label_id).all()
    return combined, len(accepted)
def train_and_validate(method, pct):
    stem = f'{method}_{pct}pct'
    metrics_path = OUT / f'{stem}_validation_metrics.csv'
    if metrics_path.exists():
        print(f'Reusing completed run: {stem}', flush=True)
        return pd.read_csv(metrics_path)
    df, added = approved_training(method, pct)
    print(f'Starting {stem}: {len(df)} total, {added} augmented', flush=True)
    validation = pd.read_csv(ROOT / 'data/processed/astd_validation.csv')
    validation_labels = validation.label.map(LABEL2ID)
    assert validation_labels.notna().all(), 'Unexpected validation label'
    if 'label_id' in validation:
        assert validation_labels.eq(validation.label_id).all()
    validation['label_id'] = validation_labels.astype(int)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    cuda = device.type == 'cuda'
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if cuda:
        torch.cuda.manual_seed_all(SEED)
    train_loader = DataLoader(
        TweetDS(df), batch_size=1, shuffle=True, collate_fn=collate,
        generator=torch.Generator().manual_seed(SEED), pin_memory=cuda,
    )
    val_loader = DataLoader(TweetDS(validation), batch_size=8, collate_fn=collate)
    model = AutoModelForSequenceClassification.from_pretrained(
        CLASSIFIER, num_labels=4, label2id=LABEL2ID,
        id2label={i: label for label, i in LABEL2ID.items()},
    ).to(device)
    model.config.use_cache = False
    model.gradient_checkpointing_enable()
    counts = df.label_id.value_counts().reindex(range(4), fill_value=0)
    assert (counts > 0).all()
    class_weights = torch.tensor(len(df) / (4 * counts.to_numpy()), dtype=torch.float32, device=device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
    scaler = torch.amp.GradScaler('cuda', init_scale=8192, enabled=cuda)
    best, stale, history = -1, 0, []
    for epoch in range(1, 4):
        model.train()
        weighted_total = weight_total = 0.0
        iterator = iter(train_loader)
        while group := list(islice(iterator, 2)):
            optimizer.zero_grad(set_to_none=True)
            labels = [batch.pop('labels').to(device) for batch in group]
            weights = [class_weights[y] for y in labels]
            denominator = sum(w.sum() for w in weights)
            for batch, y, w in zip(group, labels, weights):
                inputs = {k: v.to(device) for k, v in batch.items()}
                with torch.autocast(device.type, dtype=torch.float16, enabled=cuda):
                    logits = model(**inputs).logits
                    numerator = (F.cross_entropy(logits.float(), y, reduction='none') * w).sum()
                scaler.scale(numerator / denominator).backward()
                weighted_total += numerator.detach().item()
            weight_total += denominator.item()
            scaler.unscale_(optimizer)
            gradients = [p.grad for p in model.parameters() if p.grad is not None]
            if torch.stack([g.isfinite().all() for g in gradients]).all().item():
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0, error_if_nonfinite=True)
                scaler.step(optimizer)
            elif cuda:
                scaler.step(optimizer)
            scaler.update()
        metrics, matrix = evaluate(model, val_loader, device)
        history.append({
            'method': method, 'fraction': pct, 'epoch': epoch,
            'training_loss': weighted_total / weight_total, 'augmented_rows': added, **metrics,
        })
        print(history[-1], flush=True)
        if metrics['macro_f1'] > best:
            best, stale = metrics['macro_f1'], 0
            pd.DataFrame(matrix, index=LABELS, columns=LABELS).to_csv(OUT / f'{stem}_validation_confusion.csv')
            torch.save({k: v.detach().cpu() for k, v in model.state_dict().items()}, OUT / f'{stem}_best.pt')
        else:
            stale += 1
            if stale >= 2:
                break
    result = pd.DataFrame(history)
    temporary = metrics_path.with_suffix('.tmp')
    result.to_csv(temporary, index=False)
    temporary.replace(metrics_path)
    del optimizer, model
    gc.collect()
    if cuda:
        torch.cuda.empty_cache()
    return result
for name in ('translator_forward', 'translator_backward', 'mlm_model'):
    globals().pop(name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
for pct in FRACTIONS:
    for method in ('back_translation', 'mlm'):
        display(train_and_validate(method, pct))

{'method': 'back_translation', 'fraction': 10, 'epoch': 1, 'training_loss': 2.0979880183150414, 'augmented_rows': 589, 'accuracy': 0.6690035052578869, 'macro_precision': 0.16725087631447172, 'macro_recall': 0.25, 'macro_f1': 0.2004200420042004, 'weighted_f1': 0.5363268424989719}
{'method': 'back_translation', 'fraction': 10, 'epoch': 2, 'training_loss': 2.6106129456146556, 'augmented_rows': 589, 'accuracy': 0.6690035052578869, 'macro_precision': 0.16725087631447172, 'macro_recall': 0.25, 'macro_f1': 0.2004200420042004, 'weighted_f1': 0.5363268424989719}
{'method': 'back_translation', 'fraction': 10, 'epoch': 3, 'training_loss': 2.655595344695397, 'augmented_rows': 589, 'accuracy': 0.6690035052578869, 'macro_precision': 0.16725087631447172, 'macro_recall': 0.25, 'macro_f1': 0.2004200420042004, 'weighted_f1': 0.5363268424989719}
{'method': 'mlm', 'fraction': 10, 'epoch': 1, 'training_loss': 2.037161107831313, 'augmented_rows': 599, 'accuracy': 0.6765147721582374, 'macro_precision': 0.278